# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
from dotenv import load_dotenv
load_dotenv(r"C:\Users\ufift\deploying-ai\05_src\.secrets", override=True)

True

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
# Load PDF
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [4]:
# Join the pages 
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


In [5]:
print(document_text[:1000])

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age of

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import os
import json
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

# Define the structured output schema
tone_name = "Formal Academic Writing"

class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [7]:
user_prompt = f"""
Analyze the following article and return structured output.

ARTICLE TEXT:
{document_text[:15000]}
"""

In [8]:
#Keep instructions and context separate

developer_instructions = f"""
You are an expert reading assistant for AI professionals.

Return ONLY valid JSON.
Do NOT include:
- explanations
- extra text
- markdown
- backticks

Output must start with {{ and end with }}.

Use exactly these keys:
Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens

Requirements:
- Relevance: max one paragraph
- Summary: max 1000 tokens
- Tone: {tone_name}
- Do not invent information
- Set InputTokens and OutputTokens to 0
"""

In [9]:
# Generate response
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=developer_instructions,
    input=user_prompt
)

In [10]:
print(response.output_text)

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article explores the importance of self-knowledge in achieving personal and professional success, emphasizing that individuals must take responsibility for managing their own careers in the knowledge economy.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker argues that success in the modern knowledge economy hinges on self-awareness—understanding one's strengths, weaknesses, values, and work style. He posits that unlike in the past, where roles were predefined, contemporary workers must act as their own chief executives, navigating their careers with autonomy and intention. Drucker introduces the concept of 'feedback analysis' as a practical method for individuals to identify their strengths by comparing expected outcomes of decisions with the actual results over time. He advises focusing on personal strengths rather than attempting to improve weaknesses, advocating for enhanced performance thr

In [11]:
data = json.loads(response.output_text)

# replace token counts with actual usage from response
data["InputTokens"] = response.usage.input_tokens
data["OutputTokens"] = response.usage.output_tokens

final_result = ArticleAnalysis(**data)

In [12]:
# Print final result output
print(final_result)


Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article explores the importance of self-knowledge in achieving personal and professional success, emphasizing that individuals must take responsibility for managing their own careers in the knowledge economy.' Summary="In 'Managing Oneself,' Peter F. Drucker argues that success in the modern knowledge economy hinges on self-awareness—understanding one's strengths, weaknesses, values, and work style. He posits that unlike in the past, where roles were predefined, contemporary workers must act as their own chief executives, navigating their careers with autonomy and intention. Drucker introduces the concept of 'feedback analysis' as a practical method for individuals to identify their strengths by comparing expected outcomes of decisions with the actual results over time. He advises focusing on personal strengths rather than attempting to improve weaknesses, advocating for enhanced performance through self-discovery and d

In [13]:
print(final_result.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article explores the importance of self-knowledge in achieving personal and professional success, emphasizing that individuals must take responsibility for managing their own careers in the knowledge economy.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker argues that success in the modern knowledge economy hinges on self-awareness—understanding one's strengths, weaknesses, values, and work style. He posits that unlike in the past, where roles were predefined, contemporary workers must act as their own chief executives, navigating their careers with autonomy and intention. Drucker introduces the concept of 'feedback analysis' as a practical method for individuals to identify their strengths by comparing expected outcomes of decisions with the actual results over time. He advises focusing on personal strengths rather than attempting to improve weaknesses, advocating for enhanced performance thr

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Import required libraries and functions
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)
# Create test case, limiting input text to 15000 for token safety
test_case = LLMTestCase(input=document_text[:15000], actual_output= final_result.Summary)
metric = SummarizationMetric(
    threshold=0.5,
    model= model,
    assessment_questions=[
         "Does the summary accurately describe the main purpose of the article?",
        "Does the summary include the most important ideas from the original text?",
        "Does the summary avoid adding information that is not present in the source?",
        "Does the summary preserve the meaning of the original article?",
        "Does the summary clearly explain the author's main argument?"
    ]
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.7692307692307693, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.77 because the summary introduces extra information that is not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, and the summary captures the essence of the original text well., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Rep

✓ Evaluation completed 🎉! (time taken: 15.25s | token cost: 0.0020787 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Summarization', threshold=0.5, success=True, score=0.7692307692307693, reason='The score is 0.77 because the summary introduces extra information that is not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, and the summary captures the essence of the original text well.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0020787, verbose_logs='Truths (limit=None):\n[\n    "Success in the knowledge economy comes to those who know themselves, their strengths, their values, and how they best perform.",\n    "Companies today are not managing their employees\' careers; knowledge workers must manage their own careers.",\n    "Individuals must cultivate a deep understanding of themselves to succeed in their careers.",\n    "Feedback analysis is a method to identify one\'s streng

In [15]:
results = evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.5384615384615384, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.54 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, indicating that the core ideas are preserved., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Rep

✓ Evaluation completed 🎉! (time taken: 18.35s | token cost: 0.0020593499999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [16]:
score = results.test_results[0].metrics_data[0].score
reason = results.test_results[0].metrics_data[0].reason
success = results.test_results[0].metrics_data[0].success

print("Score:", score)
print("Reason:", reason)
print("Success:", success)

Score: 0.5384615384615384
Reason: The score is 0.54 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, indicating that the core ideas are preserved.
Success: True


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Create a prompt to improve summary keeping evaluation feedback in front
enhancement_prompt = f"""
You are improving a summary based on evaluation feedback.

Original article:
{document_text[:15000]}

Previous summary:
{final_result.Summary}

Evaluation score:
{score}

Evaluation reason:
{reason}

Evaluation success:
{success}

Task:
Rewrite the summary to improve factual accuracy, coverage, clarity, and completeness.

Rules:
- Do not add unsupported information.
- Keep the same tone: {final_result.Tone}
- Make the summary concise but complete.
- Include only information supported by the original article.
- Return only valid JSON with exactly these keys:
Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens

Set InputTokens and OutputTokens to 0.
"""

In [ ]:
# Call LLM to generate summary after enhancement
enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=developer_instructions,
    input=enhancement_prompt
)

print(enhanced_response.output_text)

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "Drucker emphasizes that in the modern knowledge economy, individuals must understand their strengths, weaknesses, and values to effectively manage their careers and achieve excellence.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker asserts that success in today's knowledge economy depends on a thorough self-understanding, encompassing one's strengths, weaknesses, values, and preferred work styles. He contends that contemporary employees must act as their own chief executives, taking charge of their careers rather than relying on organizations to manage them. Central to this self-management is the method of 'feedback analysis,' whereby individuals assess their expected outcomes against actual results to identify genuine strengths. Drucker advises focusing on enhancing these strengths rather than attempting to improve weaknesses. Identifying an appropriate work environment that aligns with one's cap

In [19]:
enhanced_data = json.loads(enhanced_response.output_text)

enhanced_data["InputTokens"] = enhanced_response.usage.input_tokens
enhanced_data["OutputTokens"] = enhanced_response.usage.output_tokens

enhanced_result = ArticleAnalysis(**enhanced_data)

print(enhanced_result.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "Drucker emphasizes that in the modern knowledge economy, individuals must understand their strengths, weaknesses, and values to effectively manage their careers and achieve excellence.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker asserts that success in today's knowledge economy depends on a thorough self-understanding, encompassing one's strengths, weaknesses, values, and preferred work styles. He contends that contemporary employees must act as their own chief executives, taking charge of their careers rather than relying on organizations to manage them. Central to this self-management is the method of 'feedback analysis,' whereby individuals assess their expected outcomes against actual results to identify genuine strengths. Drucker advises focusing on enhancing these strengths rather than attempting to improve weaknesses. Identifying an appropriate work environment that aligns with one's cap

In [20]:
# Create new test case for enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text[:15000],
    actual_output=enhanced_result.Summary
)

# Run evaluation again
enhanced_results = evaluate(
    test_cases=[enhanced_test_case],
    metrics=[metric]
)

# Extract new scores
enhanced_score = enhanced_results.test_results[0].metrics_data[0].score
enhanced_reason = enhanced_results.test_results[0].metrics_data[0].reason

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the summary introduces multiple pieces of extra information that are not present in the original text, which misrepresents the content and intent of the original material., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCI

✓ Evaluation completed 🎉! (time taken: 13.7s | token cost: 0.0019880999999999996 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
# Create to new enhancment prompt with more strict criteria to improve the evaluation summary metrics
# after the first enhancment 
enhancement_prompt_ = f"""
Rewrite the summary using ONLY the original article.

Original article:
{document_text[:15000]}

Previous summary:
{final_result.Summary}

Evaluation feedback:
Score: {score}
Reason: {reason}

Important instructions:
- Do NOT add new information.
- Do NOT infer beyond the article.
- Do NOT mention ideas unless they are clearly stated in the article.
- Remove unsupported details from the previous summary.
- Focus only on Drucker's main points:
  1. Knowledge workers must manage themselves.
  2. People should identify their strengths through feedback analysis.
  3. People should understand how they work, learn, and perform.
  4. Values and work environment matter.
  5. People should focus on strengths instead of weaknesses.
- Keep the summary concise and factual.

Return ONLY valid JSON in this exact format:
{{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "The article is relevant because it explains how individuals can manage their careers by understanding their strengths, values, and working style.",
  "Summary": "...",
  "Tone": "{final_result.Tone}",
  "InputTokens": 0,
  "OutputTokens": 0
}}
"""

In [23]:
enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=developer_instructions,
    input=enhancement_prompt_
)

print(enhanced_response.output_text)

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "The article is relevant because it explains how individuals can manage their careers by understanding their strengths, values, and working style.",
  "Summary": "In 'Managing Oneself,' Peter F. Drucker emphasizes that success in the knowledge economy is contingent upon self-awareness, which encompasses understanding one's strengths, weaknesses, values, and preferred work style. He asserts that, unlike in previous eras where career paths were predetermined, contemporary knowledge workers must assume the role of their own chief executives and actively steer their professional trajectories. Drucker introduces 'feedback analysis' as a method for individuals to pinpoint their strengths by comparing anticipated outcomes of their decisions with the actual results over time. He advises prioritizing the cultivation of personal strengths instead of attempting to ameliorate weaknesses. Furthermore, Drucker highlights

In [24]:
enhanced_data = json.loads(enhanced_response.output_text)

enhanced_data["InputTokens"] = enhanced_response.usage.input_tokens
enhanced_data["OutputTokens"] = enhanced_response.usage.output_tokens

enhanced_result = ArticleAnalysis(**enhanced_data)

print(enhanced_result.Summary)

In 'Managing Oneself,' Peter F. Drucker emphasizes that success in the knowledge economy is contingent upon self-awareness, which encompasses understanding one's strengths, weaknesses, values, and preferred work style. He asserts that, unlike in previous eras where career paths were predetermined, contemporary knowledge workers must assume the role of their own chief executives and actively steer their professional trajectories. Drucker introduces 'feedback analysis' as a method for individuals to pinpoint their strengths by comparing anticipated outcomes of their decisions with the actual results over time. He advises prioritizing the cultivation of personal strengths instead of attempting to ameliorate weaknesses. Furthermore, Drucker highlights the importance of aligning one's work environment with personal values and capabilities, which can lead to enhanced contributions in the workplace. He also underscores the significance of interpersonal skills and manners in collaborative sett

In [ ]:
# Create another test case after giving second enhancment propmt 
enhanced_test_case = LLMTestCase(
    input=document_text[:15000],
    actual_output=enhanced_result.Summary
)

enhanced_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    include_reason=True,
    assessment_questions=[
        "Does the summary accurately describe the main purpose of the article?",
        "Does the summary include the most important ideas from the original text?",
        "Does the summary avoid adding information that is not present in the source?",
        "Does the summary preserve the meaning of the original article?",
        "Does the summary clearly explain the author's main argument?"
    ]
)

enhanced_results = evaluate(
    test_cases=[enhanced_test_case],
    metrics=[enhanced_metric]
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8333333333333334, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.83 because the summary includes extra information that was not present in the original text, which may lead to misinterpretation of the original content. However, there are no contradictions, and the summary captures the main ideas effectively., error: None)

For test case:

  - input: www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KT

✓ Evaluation completed 🎉! (time taken: 16.99s | token cost: 0.00197895 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [26]:
enhanced_score = enhanced_results.test_results[0].metrics_data[0].score
enhanced_reason = enhanced_results.test_results[0].metrics_data[0].reason
enhanced_success = enhanced_results.test_results[0].metrics_data[0].success

print("Enhanced Score:", enhanced_score)
print("Enhanced Reason:", enhanced_reason)
print("Enhanced Success:", enhanced_success)

Enhanced Score: 0.8333333333333334
Enhanced Reason: The score is 0.83 because the summary includes extra information that was not present in the original text, which may lead to misinterpretation of the original content. However, there are no contradictions, and the summary captures the main ideas effectively.
Enhanced Success: True


The summary recieved a score of 0.54 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation of the original message.
The first enhancment recived a socre of 0.00 because it added several unsupported details.
After revising the enhancement prompt to make it more strict to generate summary based on the given article, 
the scroe improved to 0.83.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
